### RAG Pipeline- Data Ingestion to Vector DB Pipeline

In [10]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\Sudarshan Ghogare\AppData\Local\Temp\ipykernel_25772\2096081597.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [11]:
### Read all the pdfs inside the directory
from pathlib import Path
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f" Error: {e}")
            
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")
        
        

Found 2 PDF files to process

Processing: MONALI GHOGARE RESUME.pdf
 Loaded 1 pages

Processing: MonaliGokulGhogare.pdf
 Loaded 1 pages

Total documents loaded: 2


In [12]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-08T00:20:48+05:30', 'author': 'sudarshan ghogare', 'moddate': '2026-09-08T00:20:48+05:30', 'source': '..\\data\\pdf\\MONALI GHOGARE RESUME.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'MONALI GHOGARE RESUME.pdf', 'file_type': 'pdf'}, page_content='MONALI GHOGARE \n• Phone: +91 9209448691 \n• Email: monalighogare14@gmail.com \n• LinkedIn Profile \n• GitHub Profile \nPROFESSIONAL SUMMARY \nEntry-level Python Full Stack Developer with hands -on experience in Python, FastAPI,  React.js, \nJavaScript, REST APIs, and PostgreSQL. Skilled in developing responsive web applications, building CURD -\nbased APIs, integrating databases, and creating reusable frontend components. \nTECHNICAL SKILLS \n• Programming Languages: Python, JavaScript, Java \n• Frontend: HTML, CSS, JavaScript, React.js \n• Backend: FastAPI, REST APIs, CRUD Operations, Authentication \n•

### Text splitting get into chunks



In [13]:
def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    """split documents into smaller chunk for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs
    

In [14]:
chunks = split_documents(all_pdf_documents)
chunks

split 2 documents into 8 chunks

Example chunk:
Content: MONALI GHOGARE 
• Phone: +91 9209448691 
• Email: monalighogare14@gmail.com 
• LinkedIn Profile 
• GitHub Profile 
PROFESSIONAL SUMMARY 
Entry-level Python Full Stack Developer with hands -on experien...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-08T00:20:48+05:30', 'author': 'sudarshan ghogare', 'moddate': '2026-09-08T00:20:48+05:30', 'source': '..\\data\\pdf\\MONALI GHOGARE RESUME.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'MONALI GHOGARE RESUME.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-09-08T00:20:48+05:30', 'author': 'sudarshan ghogare', 'moddate': '2026-09-08T00:20:48+05:30', 'source': '..\\data\\pdf\\MONALI GHOGARE RESUME.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'MONALI GHOGARE RESUME.pdf', 'file_type': 'pdf'}, page_content='MONALI GHOGARE \n• Phone: +91 9209448691 \n• Email: monalighogare14@gmail.com \n• LinkedIn Profile \n• GitHub Profile \nPROFESSIONAL SUMMARY \nEntry-level Python Full Stack Developer with hands -on experience in Python, FastAPI,  React.js, \nJavaScript, REST APIs, and PostgreSQL. Skilled in developing responsive web applications, building CURD -\nbased APIs, integrating databases, and creating reusable frontend components. \nTECHNICAL SKILLS \n• Programming Languages: Python, JavaScript, Java \n• Frontend: HTML, CSS, JavaScript, React.js \n• Backend: FastAPI, REST APIs, CRUD Operations, Authentication \n•

### Embedding and vector store DB

In [15]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embedding
        """

        self.model_name = model_name
        self.model = None

        # Load the model
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""

        print(f"Loading model: {self.model_name}")

        self.model = SentenceTransformer(self.model_name)

        print("Model loaded successfully!")


# Initialize the embedding manager
embedding_manager = EmbeddingManager()

embedding_manager

Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1743.56it/s]


Model loaded successfully!


### Vector store

In [17]:

import os
import chromadb


class VectoreStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized: {self.collection_name}")
            print(f"Existing documents: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: list,
        embeddings: list,
        metadatas: list = None,
        ids: list = None
    ):
        """Add documents and their embeddings to ChromaDB."""
        try:
            if not documents:
                print("No documents to add.")
                return

            if len(documents) != len(embeddings):
                raise ValueError(
                    "Documents and embeddings must have the same length."
                )

            if metadatas is not None and len(metadatas) != len(documents):
                raise ValueError(
                    "Metadata count must match document count."
                )

            if ids is None:
                ids = [f"doc_{i}" for i in range(len(documents))]

            if len(ids) != len(documents):
                raise ValueError("ID count must match document count.")

            self.collection.add(
                documents=documents,
                embeddings=embeddings,
                metadatas=metadatas,
                ids=ids
            )

            print(f"Successfully added {len(documents)} documents.")

        except Exception as e:
            print(f"Error adding documents: {e}")
            raise


# Create the vector store object
vectorestore = VectoreStore()

Vector store initialized: pdf_documents
Existing documents: 0


In [18]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Set your text files folder
data_path = Path("../data/text_files")

# 2. Load text documents
loader = DirectoryLoader(
    str(data_path),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print("Documents loaded:", len(documents))

# 3. Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Chunks created:", len(chunks))

Documents loaded: 2
Chunks created: 4


In [19]:
chunks

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Introduction\nMachine Learning is a technology that enables computers to learn patterns from data and use those patterns to make predictions or decisions.'),
 Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='-Types of Machine Learning\n1.Supervised Learning\nModel learns from labeled data.\nExample: Predicting house prices, student results.\n2.Unsupervised Learning\nModel finds patterns or groups in data without labels.\nExample: Customer segmentation.\n3.Reinforcement Learning\nModel learns through rewards and penalties.\nExample: Game-playing AI and robotics.'),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n    Python is a simple, powerful, and versatile programming language used for web development, software development, data analysis, AI, ML, and automation.'),
 

In [20]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

data_path = Path("../data/text_files")

loader = DirectoryLoader(
    str(data_path),
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()
print("Documents loaded:", len(documents))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)
print("Chunks created:", len(chunks))

Documents loaded: 2
Chunks created: 4


### Retrival Pipeline from Vector Store

In [21]:
def search_documents(self, query_embedding: list, n_results: int = 3):
    results = self.collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    retrieved_documents = []

    for i, document in enumerate(results["documents"][0]):
        retrieved_documents.append({
            "document": document,
            "metadata": results["metadatas"][0][i],
            "id": results["ids"][0][i]
        })

    return retrieved_documents

In [22]:
vectorestore = VectoreStore()

Vector store initialized: pdf_documents
Existing documents: 0


### Integration VectorDB Context pipline With LLM output

In [23]:
### Simple RAG Pipeline with groq LLM 
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env file")

client = Groq(api_key=api_key)

print("Groq client initialized successfully.")


Groq client initialized successfully.


In [24]:
def simple_rag(question: str, top_k: int = 3):

    # 1. Convert the question into an embedding
    query_embedding = embedding_manager.generate_embeddings(
        [question]
    )

    # 2. Retrieve relevant chunks from ChromaDB
    retrieved_docs = vectorestore.search_documents(
        query_embedding=query_embedding,
        n_results=top_k
    )

    # 3. Check whether any documents were retrieved
    if not retrieved_docs:
        return "No relevant documents found in the vector store."

    # 4. Combine retrieved chunks into context
    context = "\n\n".join(
        item["document"] for item in retrieved_docs
    )

    # 5. Create the prompt
    prompt = f"""
    Answer the question using only the context provided below.

    If the answer is not available in the context,
    say "I don't know based on the provided documents."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """

    # 6. Send the context and question to Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful RAG assistant. "
                           "Answer using the provided context only."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    # 7. Return the generated answer
    answer = response.choices[0].message.content

    return answer

In [25]:
from langchain_huggingface import HuggingFaceEmbeddings

class EmbeddingManager:
    def __init__(self):
        self.embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

    def generate_embeddings(self, texts: list):
        return self.embedding_model.embed_documents(texts)

In [26]:
embedding_manager = EmbeddingManager()

print("Embedding manager initialized successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2191.17it/s]


Embedding manager initialized successfully!


In [27]:
vectorestore = VectoreStore()

print("Documents in ChromaDB:", vectorestore.collection.count())

Vector store initialized: pdf_documents
Existing documents: 0
Documents in ChromaDB: 0


In [34]:
from types import MethodType

def search_documents(self, query_embedding: list, n_results: int = 3):
    total = self.collection.count()

    if total == 0:
        print("Vector store is empty. Add documents first.")
        return []

    results = self.collection.query(
        query_embeddings=query_embedding,
        n_results=min(n_results, total)
    )

    retrieved_documents = []

    for i, document in enumerate(results["documents"][0]):
        retrieved_documents.append({
            "document": document,
            "metadata": results["metadatas"][0][i],
            "id": results["ids"][0][i]
        })

    return retrieved_documents


vectorestore.search_documents = MethodType(search_documents, vectorestore)

print("search_documents method added successfully!")

search_documents method added successfully!


In [35]:
question = "What is machine learning?"

query_embedding = embedding_manager.generate_embeddings([question])

retrieved_docs = vectorestore.search_documents(
    query_embedding=query_embedding,
    n_results=3
)

print("Retrieved chunks:", len(retrieved_docs))

for i, item in enumerate(retrieved_docs, start=1):
    print(f"\n--- Chunk {i} ---")
    print(item["document"])

Vector store is empty. Add documents first.
Retrieved chunks: 0


### Retrivel pipeline form vectore store

In [36]:
class RAGRetriver:
    """Retrieves relevant documents from a vector store."""

    def __init__(self, embedding_manager, vectorstore):
        self.embedding_manager = embedding_manager
        self.vectorstore = vectorstore

    def retrieve(self, question: str, top_k: int = 3):
        # Step 1: Convert the question into an embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [question]
        )

        # Step 2: Search the vector store
        retrieved_docs = self.vectorstore.search_documents(
            query_embedding=query_embedding,
            n_results=top_k
        )

        # Step 3: Return retrieved documents
        return retrieved_docs

In [37]:
rag_retriever = RAGRetriver(
    embedding_manager=embedding_manager,
    vectorstore=vectorestore
)

print("RAG retriever initialized successfully!")

RAG retriever initialized successfully!


In [38]:
question = "What is machine learning?"

retrieved_docs = rag_retriever.retrieve(
    question=question,
    top_k=3
)

print("Retrieved chunks:", len(retrieved_docs))

Vector store is empty. Add documents first.
Retrieved chunks: 0


In [39]:
for i, item in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Chunk {i} ---")
    print("Document:", item["document"])
    print("Metadata:", item["metadata"])
    print("ID:", item["id"])

In [41]:
def simple_rag(question: str, top_k: int = 3):

    # Step 1: Retrieve relevant documents
    retrieved_docs = rag_retriever.retrieve(
        question=question,
        top_k=top_k
    )

    if not retrieved_docs:
        return "No relevant documents found in the vector store."

    # Step 2: Prepare context
    context = "\n\n".join(
        item["document"] for item in retrieved_docs
    )

    # Step 3: Generate answer using Groq
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful RAG assistant. Answer using "
                    "only the supplied context. If the answer is not "
                    "present in the context, say you don't know."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Context:\n{context}\n\n"
                    f"Question: {question}"
                )
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [42]:
question = "What is machine learning?"

answer = simple_rag(question)

print("Question:", question)
print("\nAnswer:", answer)

Vector store is empty. Add documents first.
Question: What is machine learning?

Answer: No relevant documents found in the vector store.


### Integration Vectordb Context pipeline with LLM output